# Sales Intelligence Dashboard — Qwen3 32B Edition
**Company:** COMP_001 &nbsp;·&nbsp; **Model:** Qwen3-32B (4-bit NF4, ~20 GB VRAM) &nbsp;·&nbsp; **Query engine:** DuckDB

| Step | Cell | What it does |
|------|------|--------------|
| 1 | Install | All pip dependencies |
| 2 | Config | Sheet URL, company, colours |
| 3 | Load data | Fetch Google Sheet → clean DataFrame |
| 4 | DuckDB | Register table, build Insight JSON |
| 5 | Model | Load Qwen3-32B 4-bit NF4 |
| 6 | SQL + Analysis engine | SQL-gen → execute → **deep analysis** pipeline |
| 7 | Charts | Plotly figure builders |
| 8 | Dashboard | Gradio two-panel app — **run last** |

---
> **GPU requirement:** Qwen3-32B in 4-bit NF4 needs ~20 GB VRAM. Use an **A100 40 GB** runtime in Colab Pro+.
> **One-time setup:** Open the Google Sheet → Share → Change to *Anyone with the link* (Viewer). Then run all cells top-to-bottom.

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys
print('Installing...')
pkgs = [
    'transformers>=4.45.0', 'accelerate>=0.34.0', 'bitsandbytes>=0.43.0',
    'duckdb>=0.10.0', 'gradio>=4.20.0', 'plotly>=5.18.0',
    'pandas', 'requests', 'torch', 'gspread',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Done')
import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1e9
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {vram:.1f} GB')
    if vram < 18:
        print('WARNING: Qwen3-32B (4-bit) needs ~20 GB VRAM. Switch to A100 runtime.')
else:
    print('WARNING: No GPU detected — set Runtime → A100 GPU')

## Cell 2 — Configuration

In [ ]:
import pandas as pd
import numpy as np
import duckdb, requests, re, json, io, warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
warnings.filterwarnings('ignore')

# ── Sheet & company ────────────────────────────────────────────────────────────
SHEET_ID       = '1rZBzew7UQCcVo4PPbUKQw1YfI_SFkEXx7FS0tMqECac'
SHEET_URL      = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid=737825579'
ACTIVE_COMPANY = 'COMP_001'
TABLE_NAME     = 'sales'

# ── Model — upgraded to Qwen3-32B ─────────────────────────────────────────────
MODEL_ID       = 'Qwen/Qwen3-32B'          # Full 32B instruct model
MAX_SQL_TOKENS = 400                        # More headroom for complex SQL
MAX_ANS_TOKENS = 1200                       # Larger budget for deep analysis
TEMPERATURE    = 0.6                        # Qwen3 recommended default
TOP_P          = 0.95                       # Qwen3 recommended default
TOP_K          = 20                         # Qwen3 recommended default
TOP_N          = 5

# ── Dashboard colours (dark theme) ────────────────────────────────────────────
CLR_BG    = '#0f1117'
CLR_CARD  = '#1a1d27'
CLR_GREEN = '#22c55e'
CLR_RED   = '#ef4444'
CLR_BLUE  = '#3b82f6'
CLR_AMBER = '#f59e0b'
CLR_TEXT  = '#e2e8f0'
CLR_MUTED = '#64748b'

print('Config OK — company:', ACTIVE_COMPANY)
print('Model    :', MODEL_ID)

## Cell 3 — Load & preprocess Google Sheet

In [ ]:
from google.colab import auth
from google.auth import default
import gspread

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)


def parse_sundries(s):
    """Parse 'Sundries: V-1042 Transport=Rs.200 | V-1055 Discount=-Rs.300'
    Returns {"Transport": 200.0, "Discount": -300.0}"""
    result = {}
    if not isinstance(s, str) or not s.strip():
        return result
    body = re.sub(r'^Sundries:\s*', '', s.strip())
    for part in body.split('|'):
        m = re.search(r'([A-Za-z][A-Za-z\s]+?)\s*=\s*(-?)(?:Rs\.|\u20b9)([\d,.]+)', part)
        if m:
            result[m.group(1).strip()] = (-1 if m.group(2) == '-' else 1) * float(m.group(3).replace(',', ''))
    return result


def load_sheet(sheet_id, company_id):
    print('Fetching sheet via authenticated Google API...')
    spreadsheet = gc.open_by_key(sheet_id)
    worksheet   = spreadsheet.get_worksheet(0)
    data        = worksheet.get_all_records()
    df          = pd.DataFrame(data)

    df.columns = df.columns.str.strip()
    print(f'  Raw rows   : {len(df)}')
    print(f'  Columns    : {df.columns.tolist()}')

    if 'company_id' not in df.columns:
        match = [c for c in df.columns if c.lower() == 'company_id']
        if match:
            df = df.rename(columns={match[0]: 'company_id'})
        else:
            raise Exception(f"'company_id' column not found. Available: {df.columns.tolist()}")

    df = df[df['company_id'] == company_id].copy().reset_index(drop=True)
    print(f'  After filter ({company_id}): {len(df)} rows')

    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

    for col in ['Qty', 'Rate', 'Amt']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['Sundries'] = df['Sundries'].fillna('')
    parsed    = df['Sundries'].apply(parse_sundries)
    sundry_df = pd.json_normalize(parsed).fillna(0)
    sundry_df.columns = [f'sundry_{c}' for c in sundry_df.columns]
    df = pd.concat([df, sundry_df], axis=1)
    df['sundry_total'] = sundry_df.sum(axis=1) if len(sundry_df.columns) else 0

    df['year']        = df['Date'].dt.year
    df['month']       = df['Date'].dt.month
    df['month_name']  = df['Date'].dt.strftime('%b %Y')
    df['quarter']     = df['Date'].dt.quarter
    df['week']        = df['Date'].dt.isocalendar().week.astype(int)
    df['day_of_week'] = df['Date'].dt.day_name()

    print(f'  Dates      : {df["Date"].min().date()} to {df["Date"].max().date()}')
    print(f'  Revenue    : Rs.{df["Amt"].sum():,.0f}')
    sundry_types = [c.replace('sundry_', '') for c in df.columns
                    if c.startswith('sundry_') and c != 'sundry_total']
    print(f'  Sundry cols: {sundry_types}')
    return df


DF = load_sheet(SHEET_ID, ACTIVE_COMPANY)
print(f'Dataset ready — {len(DF)} rows, {len(DF.columns)} columns')

## Cell 4 — DuckDB registration + Insight JSON

In [ ]:
CON = duckdb.connect(':memory:')
CON.register(TABLE_NAME, DF)

t = CON.execute(f'SELECT COUNT(*) as n, SUM(Amt) as rev FROM {TABLE_NAME}').fetchdf()
print(f'DuckDB live — {t["n"].iloc[0]} rows, Rs.{t["rev"].iloc[0]:,.0f} total revenue')


def build_schema_ctx(df):
    lines = []
    for col in df.columns:
        samples = ', '.join(str(v) for v in df[col].dropna().unique()[:3])
        lines.append(f'  {col} ({df[col].dtype}) -- e.g. {samples}')
    return '\n'.join(lines)

SCHEMA_CTX = build_schema_ctx(DF)

# ── Sundry column catalogue (injected into SQL prompt) ─────────────────────────
SUNDRY_COLS = [c for c in DF.columns if c.startswith('sundry_') and c != 'sundry_total']
SUNDRY_CTX  = ', '.join(SUNDRY_COLS) if SUNDRY_COLS else 'none'


def build_insight(df):
    total = df['Amt'].sum()

    monthly = (df.groupby(['year', 'month', 'month_name'])['Amt']
               .sum().reset_index().sort_values(['year', 'month']))
    monthly['MoM'] = monthly['Amt'].pct_change() * 100

    curr_rev = monthly['Amt'].iloc[-1] if len(monthly) >= 1 else 0
    prev_rev = monthly['Amt'].iloc[-2] if len(monthly) >= 2 else 0
    mom_pct  = round((curr_rev - prev_rev) / prev_rev * 100, 1) if prev_rev else None
    curr_mon = monthly['month_name'].iloc[-1] if len(monthly) else 'N/A'

    top_items = (df.groupby('Item')['Amt'].sum()
                 .sort_values(ascending=False).head(TOP_N).reset_index())
    top_items['pct'] = (top_items['Amt'] / total * 100).round(1)

    top_parties = (df.groupby('Party')['Amt'].sum()
                   .sort_values(ascending=False).head(TOP_N).reset_index())
    top_parties['pct'] = (top_parties['Amt'] / total * 100).round(1)

    last_ord = df.groupby('Party')['Date'].max().reset_index()
    last_ord['days_since'] = (df['Date'].max() - last_ord['Date']).dt.days

    cutoff = df['Date'].max() - pd.Timedelta(days=30)
    rec    = df[df['Date'] >= cutoff]['Amt'].sum()
    pri    = df[(df['Date'] >= cutoff - pd.Timedelta(days=30)) & (df['Date'] < cutoff)]['Amt'].sum()
    mom30  = round((rec - pri) / pri * 100, 1) if pri else None

    scols    = [c for c in df.columns if c.startswith('sundry_') and c != 'sundry_total']
    sundries = {c.replace('sundry_', ''): round(df[c].sum(), 2) for c in scols if df[c].sum() != 0}

    def fmt(v):
        if v >= 1e7:  return f'Rs.{v / 1e7:.2f}Cr'
        if v >= 1e5:  return f'Rs.{v / 1e5:.1f}L'
        return f'Rs.{v:,.0f}'

    return {
        'meta': {
            'company':       df['company_name'].iloc[0],
            'company_id':    ACTIVE_COMPANY,
            'date_from':     df['Date'].min().strftime('%d/%m/%Y'),
            'date_to':       df['Date'].max().strftime('%d/%m/%Y'),
            'voucher_count': len(df),
        },
        'kpis': {
            'total_revenue':      round(total, 2),
            'total_fmt':          fmt(total),
            'current_month':      curr_mon,
            'current_month_rev':  round(curr_rev, 2),
            'current_month_fmt':  fmt(curr_rev),
            'prev_month_fmt':     fmt(prev_rev),
            'mom_pct':            mom_pct,
            'momentum_30d':       mom30,
            'momentum_30d_fmt':   fmt(abs(mom30)) if mom30 is not None else 'N/A',
            'avg_voucher':        round(df['Amt'].mean(), 2),
            'avg_voucher_fmt':    fmt(df['Amt'].mean()),
            'unique_customers':   df['Party'].nunique(),
            'unique_products':    df['Item'].nunique(),
            'voucher_count':      len(df),
        },
        'monthly':       monthly.to_dict(orient='records'),
        'top_products':  [{'rank': i + 1, 'item': r['Item'],
                           'rev_fmt': fmt(r['Amt']), 'pct': r['pct'],
                           'rev': round(r['Amt'], 2)}
                          for i, r in top_items.iterrows()],
        'top_customers': [{'rank': i + 1, 'party': r['Party'],
                           'rev_fmt': fmt(r['Amt']), 'pct': r['pct'],
                           'rev': round(r['Amt'], 2),
                           'days_since': int(last_ord.loc[last_ord['Party'] == r['Party'], 'days_since'].iloc[0])}
                          for i, r in top_parties.iterrows()],
        'sundries': sundries,
        'risks': {
            'top1_pct':     round(top_parties['pct'].iloc[0], 1) if len(top_parties) else 0,
            'top3_pct':     round(top_parties['pct'].iloc[:3].sum(), 1) if len(top_parties) >= 3 else 0,
            'inactive_30d': int((last_ord['days_since'] > 30).sum()),
        },
    }


INSIGHT = build_insight(DF)
k = INSIGHT['kpis']
print('Insight JSON built')
print(f"  Revenue       : {k['total_fmt']}")
print(f"  MoM           : {k['mom_pct']}%")
print(f"  Momentum 30d  : {k['momentum_30d']}%")
print(f"  Top product   : {INSIGHT['top_products'][0]['item']}")
print(f"  Top customer  : {INSIGHT['top_customers'][0]['party']}")
print(f"  Sundry cols   : {SUNDRY_CTX}")

## Cell 5 — Load Qwen3-32B (4-bit NF4)
> Downloads ~18 GB on first run. Subsequent runs load from Colab cache.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print(f'Loading {MODEL_ID} in 4-bit NF4...')
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,   # bfloat16 preferred for Qwen3
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print(f'Model on : {next(model.parameters()).device}')
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB '
      f'/ {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


def llm(system, user, max_tokens=MAX_ANS_TOKENS, enable_thinking=False):
    """
    Call Qwen3-32B.
    enable_thinking=True  → /think mode (slower, better reasoning, used for deep analysis)
    enable_thinking=False → /no_think mode (fast, used for SQL generation)
    """
    # Qwen3 thinking-mode toggle via chat template
    msgs = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user},
    ]
    text = tokenizer.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=True,
        # Qwen3 hard thinking toggle — passes enable_thinking into template
        enable_thinking=enable_thinking,
    )
    inp = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=max_tokens,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0][inp['input_ids'].shape[1]:]
    raw = tokenizer.decode(gen, skip_special_tokens=True).strip()

    # Strip <think>...</think> block if thinking was enabled and we only want the answer
    if enable_thinking:
        raw = re.sub(r'<think>[\s\S]*?</think>', '', raw, flags=re.IGNORECASE).strip()

    return raw


print('LLM ready')

## Cell 6 — SQL generation + execution + deep-analysis pipeline

In [ ]:
# ── SQL generation system prompt ───────────────────────────────────────────────
SQL_SYS = f"""You are a DuckDB SQL expert. Generate ONLY a valid DuckDB SQL query — no explanation, no markdown, no backticks.

TABLE NAME: {TABLE_NAME}

COLUMNS:
{SCHEMA_CTX}

SUNDRY CHARGE COLUMNS (prefixed sundry_): {SUNDRY_CTX}
  Example: total transport cost = SUM(sundry_Transport)
  Example: total discount given = SUM(sundry_Discount)
  Always use exact column name from the list above (case-sensitive).

STRICT RULES:
1. Output ONLY the raw SQL statement ending with a semicolon.
2. Always include: WHERE company_id = '{ACTIVE_COMPANY}'
3. Only quote the column named V# like this: "V#". Never quote other column names.
4. Date is a TIMESTAMP column.
   Month grouping  : STRFTIME('%Y-%m', Date) AS month
   Quarter grouping: CONCAT(STRFTIME('%Y', Date), '-Q', CAST(QUARTER(Date) AS VARCHAR)) AS quarter
   Year grouping   : STRFTIME('%Y', Date) AS year
5. For revenue: SUM(Amt). For count: COUNT(*). For top-N: ORDER BY x DESC LIMIT N.
6. If the question is unanswerable from this table, output: SELECT 'CANNOT_ANSWER' AS result;
7. Never use CTEs. Keep queries simple and flat."""


# ── Answer / analysis system prompt ───────────────────────────────────────────
ANS_SYS = """You are a Senior Sales Analyst and Chartered Accountant for an Indian SME.
You receive two inputs:
  1. SQL query results as JSON (exact, pre-calculated numbers — ground truth)
  2. Pre-computed business insight summary (KPIs, trends, risks)

YOUR JOB — do ALL of the following:
  A. ANSWER the user's specific question using exact figures from the SQL results.
  B. ANALYSE the results in business context using the insight summary.
  C. IDENTIFY patterns, risks, or opportunities surfaced by this data.
  D. Give 1-2 ACTIONABLE recommendations the business owner can act on immediately.

STRICT NUMBER RULES:
  - Always state the exact figure from the JSON first.
  - Indian notation: value >= 1,00,00,000 → Crores | value >= 1,00,000 → Lakhs | below 1 Lakh → exact Rs. with commas.
  - Never call a sub-Lakh number 'Lakhs'. Never invent or round figures not present in the data.

FORMAT:
  - Use plain prose. No XML, no code blocks, no markdown headers.
  - Structure: Answer → Analysis → Recommendation (natural flow, not labelled sections).
  - Target 150-250 words. Be specific, not generic.
  - If the question is in Hindi, respond in Hindi."""


# ── SQL helpers ────────────────────────────────────────────────────────────────
def extract_sql(raw):
    """Strip markdown fences, grab the last SELECT...; block."""
    raw = re.sub(r'```(?:sql)?\s*', '', raw.strip(), flags=re.IGNORECASE)
    raw = raw.replace('```', '').strip()
    if ';' in raw:
        raw = raw[:raw.index(';') + 1]
    idx = raw.upper().rfind('SELECT')
    if idx != -1:
        raw = raw[idx:]
    return raw.strip()


def run_sql(sql, max_rows=50):
    if not sql or not sql.upper().startswith('SELECT'):
        return None, f'Invalid SQL extracted: {sql!r}'
    try:
        df = CON.execute(sql).fetchdf()
        return df.head(max_rows), ''
    except Exception as e:
        return None, str(e)


def safe_result_json(result_df):
    """Serialise result DataFrame to JSON, converting all datetime columns safely."""
    safe = result_df.copy()
    for c in safe.columns:
        if pd.api.types.is_datetime64_any_dtype(safe[c]):
            safe[c] = safe[c].dt.strftime('%d %b %Y')
    return json.dumps(safe.to_dict(orient='records'), indent=2, default=str)


# ── Main query pipeline ────────────────────────────────────────────────────────
def query_pipeline(user_q):
    """
    1. Qwen3 (no-think, fast) writes SQL
    2. DuckDB executes; retries once on error
    3. Qwen3 (thinking mode, deep) analyses SQL results + business insight together
    Returns dict: {sql, result_df, answer, error}
    """
    # ── Step 1: generate SQL (fast, no-think mode) ─────────────────────────────
    raw_sql = llm(
        SQL_SYS,
        f'Write DuckDB SQL to answer:\n{user_q}',
        max_tokens=MAX_SQL_TOKENS,
        enable_thinking=False,   # Speed matters for SQL gen
    )
    sql = extract_sql(raw_sql)

    # ── Step 2: execute ────────────────────────────────────────────────────────
    result_df, err = run_sql(sql)

    # ── Step 3: retry once on error ────────────────────────────────────────────
    if err:
        retry_prompt = (
            f'SQL failed with error: {err}\n'
            f'Original SQL: {sql}\n'
            f'Fix and rewrite the query for: {user_q}'
        )
        sql       = extract_sql(llm(SQL_SYS, retry_prompt, max_tokens=MAX_SQL_TOKENS, enable_thinking=False))
        result_df, err = run_sql(sql)

    if err or result_df is None:
        return {
            'sql': sql, 'result_df': None,
            'answer': f'Sorry, could not retrieve data. Error: {err}',
            'error': err,
        }

    # ── Step 4: deep analysis (thinking mode, large context) ───────────────────
    result_json   = safe_result_json(result_df)
    insight_json  = json.dumps({
        'kpis':         INSIGHT['kpis'],
        'risks':        INSIGHT['risks'],
        'top_products': INSIGHT['top_products'],
        'top_customers':INSIGHT['top_customers'],
        'sundries':     INSIGHT['sundries'],
        'monthly_trend':[{'month': r['month_name'], 'revenue': r['Amt'], 'MoM_pct': r.get('MoM')}
                         for r in INSIGHT['monthly']],
    }, indent=2, default=str)

    analysis_prompt = (
        f'User asked: "{user_q}"\n\n'
        f'--- SQL QUERY RESULTS (exact numbers, ground truth) ---\n'
        f'{result_json}\n\n'
        f'--- BUSINESS INSIGHT SUMMARY (pre-computed KPIs and trends) ---\n'
        f'{insight_json}\n\n'
        f'Using both the SQL results and the insight summary above, '
        f'answer the question, analyse the business implications, and give actionable recommendations.'
    )

    answer = llm(
        ANS_SYS,
        analysis_prompt,
        max_tokens=MAX_ANS_TOKENS,
        enable_thinking=True,    # Full reasoning mode for analysis
    )

    return {'sql': sql, 'result_df': result_df, 'answer': answer, 'error': ''}


print('SQL + analysis pipeline ready')
# Smoke test
test, err = run_sql(
    f"SELECT Party, SUM(Amt) as rev FROM {TABLE_NAME} "
    f"WHERE company_id='{ACTIVE_COMPANY}' GROUP BY Party ORDER BY rev DESC LIMIT 3;"
)
if err:
    print(f'Smoke test FAILED: {err}')
else:
    print('Smoke test passed:')
    print(test.to_string(index=False))

## Cell 7 — Plotly chart builders

In [ ]:
PLOTLY_BASE = dict(
    template='plotly_dark',
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(family='JetBrains Mono, Courier New, monospace', color=CLR_TEXT, size=11),
    margin=dict(l=10, r=10, t=36, b=36),
)


def monthly_chart(insight):
    rows = insight['monthly']
    if not rows: return go.Figure()
    labels  = [r['month_name'] for r in rows]
    revenue = [r['Amt']        for r in rows]
    mom     = [r.get('MoM')    for r in rows]

    colors = []
    for i, g in enumerate(mom):
        if i == 0 or g is None or (isinstance(g, float) and np.isnan(g)):
            colors.append(CLR_GREEN)
        else:
            colors.append(CLR_GREEN if float(g) >= 0 else CLR_RED)

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Bar(
        x=labels, y=revenue, name='Revenue',
        marker_color=colors, marker_line_width=0,
        text=[f'Rs.{v / 1e5:.1f}L' for v in revenue],
        textposition='outside', textfont=dict(size=9, color=CLR_TEXT),
    ), secondary_y=False)

    mom_clean = [
        float(g) if g is not None and not (isinstance(g, float) and np.isnan(g)) else None
        for g in mom
    ]
    fig.add_trace(go.Scatter(
        x=labels, y=mom_clean, name='MoM %',
        mode='lines+markers',
        line=dict(color=CLR_BLUE, width=2), marker=dict(size=5),
        connectgaps=True,
    ), secondary_y=True)

    fig.update_layout(
        **PLOTLY_BASE, height=310, bargap=0.3,
        legend=dict(orientation='h', y=1.1, x=0, font=dict(size=9)),
        xaxis=dict(showgrid=False, tickfont=dict(size=9)),
        yaxis=dict(showgrid=True, gridcolor='#1e2535', tickformat='.2s', title='Revenue'),
        yaxis2=dict(title='MoM %', showgrid=False,
                    zeroline=True, zerolinecolor='#334155',
                    tickfont=dict(size=9, color=CLR_BLUE),
                    titlefont=dict(size=9, color=CLR_BLUE)),
    )
    return fig


def hbar_chart(items, label_key, val_key, pct_key, color_base):
    if not items: return go.Figure()
    rev    = items[::-1]
    ys     = [r[label_key] for r in rev]
    xs     = [r[val_key]   for r in rev]
    pcts   = [r[pct_key]   for r in rev]
    n      = len(rev)
    shades = [f'rgba({color_base},{0.35 + 0.13 * i:.2f})' for i in range(n)]
    fig = go.Figure(go.Bar(
        x=xs, y=ys, orientation='h',
        marker_color=shades, marker_line_width=0,
        text=[f'{p}%' for p in pcts],
        textposition='inside', textfont=dict(size=9, color='#fff'),
    ))
    fig.update_layout(
        **PLOTLY_BASE, height=230,
        xaxis=dict(showgrid=True, gridcolor='#1e2535', tickformat='.2s'),
        yaxis=dict(showgrid=False, tickfont=dict(size=9)),
    )
    return fig


def products_chart(insight):
    return hbar_chart(insight['top_products'],  'item',  'rev', 'pct', '34,197,94')

def customers_chart(insight):
    return hbar_chart(insight['top_customers'], 'party', 'rev', 'pct', '59,130,246')


FIG_MONTHLY   = monthly_chart(INSIGHT)
FIG_PRODUCTS  = products_chart(INSIGHT)
FIG_CUSTOMERS = customers_chart(INSIGHT)
print('Charts built')

## Cell 8 — Gradio dashboard
Run this cell last. Click the public share URL to open the dashboard.

In [ ]:
import gradio as gr

# ── KPI card HTML ──────────────────────────────────────────────────────────────
def kpi_card(label, value, sub='', trend=''):
    arrow = ''
    if   trend == 'up':   arrow = f'<span style="color:{CLR_GREEN}">&#9650; </span>'
    elif trend == 'down': arrow = f'<span style="color:{CLR_RED}">&#9660; </span>'
    sub_h = (f'<div style="font-size:11px;color:{CLR_GREEN};margin-top:4px;">'
             f'{arrow}{sub}</div>') if sub else ''
    return f"""
<div style="background:{CLR_CARD};border:1px solid #2d3748;border-radius:10px;
            padding:14px 16px;flex:1;min-width:130px;">
  <div style="font-size:10px;text-transform:uppercase;letter-spacing:1px;
              color:{CLR_MUTED};margin-bottom:6px;">{label}</div>
  <div style="font-size:20px;font-weight:700;color:{CLR_TEXT};
              font-family:JetBrains Mono,monospace;">{value}</div>
  {sub_h}
</div>"""


def build_kpi_html(insight):
    k   = insight['kpis']
    m   = insight['meta']
    mom = k['mom_pct']
    m30 = k['momentum_30d']
    mom_s = (f"{'+' if mom > 0 else ''}{mom}% MoM"       if mom is not None else 'N/A')
    m30_s = (f"{'+' if m30 > 0 else ''}{m30}% vs prior 30d" if m30 is not None else 'N/A')
    mom_t = ('up' if mom and mom > 0 else 'down' if mom and mom < 0 else '')
    m30_t = ('up' if m30 and m30 > 0 else 'down' if m30 and m30 < 0 else '')
    cards = (
        kpi_card('Total Revenue',  k['total_fmt'],         mom_s, mom_t) +
        kpi_card('Current Month',  k['current_month_fmt'], k['current_month']) +
        kpi_card('Momentum (30d)', k['momentum_30d_fmt'],  m30_s, m30_t) +
        kpi_card('Avg Voucher',    k['avg_voucher_fmt'],   f"{k['voucher_count']} vouchers")
    )
    return f"""
<div style="margin-bottom:14px;">
  <div style="font-size:15px;font-weight:700;color:{CLR_TEXT};letter-spacing:.5px;">
    {m['company'].upper()}
  </div>
  <div style="font-size:11px;color:{CLR_MUTED};margin-top:2px;">
    {m['date_from']} &rarr; {m['date_to']} &nbsp;&middot;&nbsp; {m['voucher_count']} transactions
  </div>
</div>
<div style="display:flex;gap:8px;flex-wrap:wrap;margin-bottom:16px;">{cards}</div>"""


KPI_HTML = build_kpi_html(INSIGHT)

# ── Suggested questions ────────────────────────────────────────────────────────
SUGGESTED = [
    'Who are my top 5 customers by revenue?',
    'Which product had the highest total sales?',
    'Which month had the highest revenue?',
    'How much discount was given in total?',
    'Customers who have not ordered in 30+ days?',
    'What is the total transport cost?',
    'Show quarterly revenue breakdown',
    'Which item has the lowest average rate?',
]

# ── Chat handler ───────────────────────────────────────────────────────────────
def handle_query(user_msg, history):
    if not user_msg.strip():
        return history or [], history or [], ''
    try:
        result = query_pipeline(user_msg)
        answer = result['answer']
    except Exception as e:
        answer = f'Sorry, an unexpected error occurred: {str(e)}'
        result = {'sql': ''}
    history = (history or []) + [(user_msg, answer)]
    sql_md  = f'```sql\n{result["sql"]}\n```' if result.get('sql') else ''
    return history, history, sql_md


# ── Gradio layout ──────────────────────────────────────────────────────────────
CSS = f"""
body,.gradio-container{{background:{CLR_BG}!important;
  font-family:'JetBrains Mono','Courier New',monospace!important;}}
.left-panel{{border-right:1px solid #1e2535;padding:18px;}}
.right-panel{{padding:18px;}}
.sec-title{{font-size:10px;text-transform:uppercase;letter-spacing:1.5px;
  color:{CLR_MUTED};margin:14px 0 6px;}}
.sug-btn{{background:#1a2035!important;border:1px solid #2d3748!important;
  color:{CLR_TEXT}!important;border-radius:8px!important;font-size:11px!important;
  padding:7px 10px!important;text-align:left!important;}}
.sug-btn:hover{{background:#2d3748!important;border-color:{CLR_BLUE}!important;}}
footer{{display:none!important;}}
"""

with gr.Blocks(css=CSS, theme=gr.themes.Base(), title='Sales Intelligence — Qwen3 32B') as demo:
    with gr.Row(equal_height=False):

        # ── LEFT: dashboard ───────────────────────────────────────────────────
        with gr.Column(scale=6, elem_classes='left-panel'):
            gr.HTML(KPI_HTML)
            gr.HTML(f'<div class="sec-title">Monthly Revenue</div>')
            gr.Plot(FIG_MONTHLY, show_label=False)
            with gr.Row():
                with gr.Column(scale=1):
                    gr.HTML(f'<div class="sec-title">Top Products</div>')
                    gr.Plot(FIG_PRODUCTS, show_label=False)
                with gr.Column(scale=1):
                    gr.HTML(f'<div class="sec-title">Top Customers</div>')
                    gr.Plot(FIG_CUSTOMERS, show_label=False)

        # ── RIGHT: AI chat ────────────────────────────────────────────────────
        with gr.Column(scale=4, elem_classes='right-panel'):
            gr.HTML(f"""
<div style="margin-bottom:14px;">
  <div style="display:flex;align-items:center;gap:8px;">
    <div style="width:8px;height:8px;background:{CLR_GREEN};border-radius:50%;"></div>
    <span style="font-size:13px;font-weight:700;color:{CLR_TEXT};">AI CA Assistant &mdash; Qwen3 32B</span>
  </div>
  <div style="font-size:11px;color:{CLR_MUTED};margin-top:2px;margin-left:16px;">
    Deep analysis &middot; English or Hindi
  </div>
</div>""")

            gr.HTML(f'<div class="sec-title" style="color:{CLR_AMBER};">Suggested Questions</div>')

            query_box = gr.Textbox(
                label='', show_label=False, lines=1, max_lines=3,
                placeholder='Ask about your sales... (English / Hindi)',
            )

            for i in range(0, len(SUGGESTED), 2):
                with gr.Row():
                    for q in SUGGESTED[i:i + 2]:
                        b = gr.Button(q, elem_classes='sug-btn', size='sm')
                        b.click(fn=lambda x=q: x, outputs=query_box)

            gr.HTML(f'<div class="sec-title" style="margin-top:14px;">Conversation</div>')

            chatbot = gr.Chatbot(
                label='', show_label=False, height=300,
                bubble_full_width=False, avatar_images=(None, '\U0001f916'),
            )

            sql_box = gr.Markdown(value='', label='Generated SQL')

            with gr.Row():
                send_btn  = gr.Button('Ask \u2192', variant='primary',   scale=3)
                clear_btn = gr.Button('Clear',       variant='secondary', scale=1)

            history_st = gr.State([])

            send_btn.click(
                fn=handle_query,
                inputs=[query_box, history_st],
                outputs=[chatbot, history_st, sql_box],
            ).then(lambda: '', outputs=query_box)

            query_box.submit(
                fn=handle_query,
                inputs=[query_box, history_st],
                outputs=[chatbot, history_st, sql_box],
            ).then(lambda: '', outputs=query_box)

            clear_btn.click(
                fn=lambda: ([], [], ''),
                outputs=[chatbot, history_st, sql_box],
            )

print('Launching dashboard...')
print('A public share URL will appear below.')
demo.launch(share=True, quiet=True)